# Similar teas, from shop catalogues

The vendor harvest in `data/vendor-catalogues/`. Used to choose a similarity method, not shipped in the app.

In [ ]:
import numpy as np
import pandas as pd

from herbatka_analysis import paths

paths.ensure()
VENDOR = paths.ROOT / "data" / "vendor-catalogues"

In [ ]:
VENDOR

## All teas in one table

The harvest is saved as one CSV file per country. This cell joins them into one table, `teas`, with one row per tea. Every file has the same columns, so the rows stack cleanly.

In [ ]:
teas = pd.concat(
    (pd.read_csv(f) for f in sorted(VENDOR.glob("teas_*.csv"))),
    ignore_index=True,
)

print(f"{len(teas):,} teas, {teas['shop'].nunique()} shops, {teas['country'].nunique()} countries")
teas.head()

## A first look at every column

Three quick checks before any real analysis.

**The column table.** One row per column of `teas`:
- `dtype` is the kind of value. `str` is text and `int64` is whole numbers. `object` here means text mixed with missing values.
- `filled` is how many teas have a value.
- `missing_%` is the share of teas with no value.
- `unique` is how many different values appear. Compare it with the number of teas. `url` is different for every tea, so it can identify a tea. `name` is not, so some names repeat.

**The `n_ingredients` summary.**
- `count` is how many teas, and `mean` is the average.
- `std` (standard deviation) is how far values typically sit from the average.
- `min` and `max` are the smallest and largest values.
- `25%`, `50%` and `75%` are *quartiles*. A quarter of teas are at or below the `25%` value, half at or below `50%`, and so on. The `50%` value is the *median*: the tea in the middle.
- When the mean is above the median, a few teas with long lists are pulling the average up.

**The value counts.** How many teas have each value of a column. `NaN` means the value is missing.

In [ ]:
overview = pd.DataFrame({
    "dtype": teas.dtypes.astype(str),
    "filled": teas.notna().sum(),
    "missing_%": (teas.isna().mean() * 100).round(1),
    "unique": teas.nunique(),
})
display(overview)

display(teas["n_ingredients"].describe())

for column in ["tea_type", "source_quality", "usable_for_similarity"]:
    display(teas[column].value_counts(dropna=False))

## Turning list columns into rows

`ingredients_english`, `flavour_families` and `quality_flags` each store a list as one piece of text, like `"Black tea leaf, Vanilla"`. You can't count items in that form.

- `split_list` cuts the text at each separator and gives one row per item. This is called *exploding* a column. Each row keeps the number of the tea it came from, so items can be counted per tea or traced back.
- `teas_per_item` counts how many **teas** have each item. It removes repeats first, so a tea that lists the same ingredient twice still counts once.
- `n_mapped` is the number of ingredients actually in `ingredients_english`. Use it instead of `n_ingredients`, which also counts dropped words such as countries and "natural flavouring".

This cell shows nothing. The sections below use what it builds.

In [ ]:
def split_list(column: str, sep: str = ",") -> pd.Series:
    items = teas[column].str.split(sep).explode().str.strip()
    return items[items.notna() & (items != "")]


def teas_per_item(items: pd.Series) -> pd.Series:
    return items.reset_index().drop_duplicates().value_counts(items.name)


ingredients = split_list("ingredients_english")
flavours = split_list("flavour_families")
flags = split_list("quality_flags", ";")

teas["n_mapped"] = ingredients.groupby(level=0).size().reindex(teas.index, fill_value=0)

## Does the shop shape the data?

**Why it matters.** The data comes from many shops, and each one writes its pages differently. If one shop were most of the data, the model would learn that shop's writing style, not tea in general.

**How to read the output.**
- The two printed lines give the share of all teas that come from the biggest shop and from the top five shops. Lower is healthier.
- The coloured table shows, for each shop, the share of its teas missing each column. White is 0% missing and dark red is 100%. Rows are sorted by missing flavours.
- The bar chart is the number of teas per shop.

**What to look for.** Red that clusters in a few rows. That means the gaps aren't random: they come from how particular shops write. In that case a filter like "keep only teas with flavours" isn't neutral. It quietly removes most of those shops.

In [ ]:
per_shop = teas["shop"].value_counts()
print(f"largest shop, {per_shop.index[0]}: {per_shop.iloc[0] / len(teas):.0%} of teas")
print(f"top 5 shops: {per_shop.head(5).sum() / len(teas):.0%} of teas")

gaps = (
    teas[["tea_type", "format", "ingredients_english", "flavour_families"]]
    .isna()
    .groupby(teas["shop"])
    .mean()
    .sort_values("flavour_families")
)
display(gaps.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

per_shop.plot.barh(figsize=(6, 9)).invert_yaxis()

## How many ingredients does a tea have?

Counted with `n_mapped`, the ingredients actually in `ingredients_english`. Not `n_ingredients`, which also counts words that were dropped from the list.

**Why it matters.** Teas are matched through shared ingredients. A tea with one ingredient can only match teas that have that same one. A tea with twelve has many more ways to match.

**How to read the output.**
- The table splits teas by `source_quality`: ingredients taken from a proper composition list, or picked out of the shop's description text. The columns mean the same as in the `n_ingredients` summary near the top.
- **Skew** measures how lopsided the counts are. 0 means balanced. A positive number means most teas have few ingredients and a small number have many: a long tail to the right. Above 1 is strongly lopsided.
- The histogram has one bar per ingredient count. Bar height is how many teas have that count.

**What to look for.** Tall bars at 0 and 1: those teas give the model almost nothing to work with. Also compare the two `source_quality` rows. If their minimums or spreads differ a lot, the two sources don't produce comparable lists.

In [ ]:
display(teas.groupby("source_quality")["n_mapped"].describe())
print(f"skew: {teas['n_mapped'].skew():.2f}")
teas["n_mapped"].plot.hist(bins=range(teas["n_mapped"].max() + 2), figsize=(7, 3))

## How are ingredients and flavours spread?

**Why it matters.** An ingredient on almost every tea tells two teas apart about as well as "both are tea". A very rare one can only link a couple of teas. The useful ones sit in between. The weighting step later decides how much each one counts.

**How to read the output.**
- **Distinct ingredients** is the size of the vocabulary: how many different ingredients appear anywhere.
- **Used by a single tea** counts ingredients that can never make two teas similar, because no other tea has them.
- A **mention** is one tea listing one ingredient. "N ingredients account for 80% of all mentions" measures concentration. A small N means a few ingredients do most of the work.
- **`share_of_teas`** is the fraction of all teas that list that item. 25% means one tea in four.
- The last table is how many teas carry each quality flag. One tea can have several flags.

In [ ]:
ingredient_teas = teas_per_item(ingredients)
mention_share = ingredient_teas.cumsum() / ingredient_teas.sum()
print(f"{len(ingredient_teas)} distinct ingredients, {(ingredient_teas == 1).sum()} used by a single tea")
print(f"{(mention_share < 0.8).sum() + 1} ingredients account for 80% of all mentions")

display((ingredient_teas.head(15) / len(teas)).to_frame("share_of_teas").style.format("{:.1%}"))
display((teas_per_item(flavours) / len(teas)).to_frame("share_of_teas").style.format("{:.1%}"))
display(teas_per_item(flags).to_frame("teas"))

### Ingredients as charts

**Left: share of teas listing each ingredient.** The same numbers as the table above, for the 30 most common ingredients. A longer bar means a more common ingredient. Check the top bars against what you know about tea: an ingredient ranked higher than it should be is often a data error, not a trend.

**Right: cumulative share of mentions.** *Cumulative* means running total. Sort the ingredients from most to least common, then walk along the list and add up their mentions as you go. The curve shows how much of the total you have after each step.
- A curve that shoots up and then goes flat means a few ingredients dominate.
- A straight diagonal line would mean every ingredient is equally common.
- The dot marks where the running total reaches 80%.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

SERIES = "#2a78d6"
chart_style = {
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#52514e",
    "axes.titlecolor": "#0b0b0b",
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 1,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "xtick.labelcolor": "#52514e",
    "ytick.labelcolor": "#52514e",
    "font.size": 9,
}

top = (ingredient_teas / len(teas)).head(30)
rank = np.arange(1, len(mention_share) + 1)
cutoff = (mention_share < 0.8).sum() + 1

with plt.rc_context(chart_style):
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 7))

    left.barh(top.index, top.to_numpy(), height=0.7, color=SERIES)
    left.invert_yaxis()
    left.grid(axis="y", visible=False)
    left.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    left.set_title("Share of teas listing each ingredient, top 30")

    right.plot(rank, mention_share.to_numpy(), color=SERIES, linewidth=2)
    right.axhline(0.8, color="#c3c2b7", linewidth=1)
    right.plot(cutoff, mention_share.iloc[cutoff - 1], "o", color=SERIES,
               markersize=8, markeredgecolor="#fcfcfb", markeredgewidth=2)
    right.annotate(f"{cutoff} of {len(mention_share)} ingredients\nmake up 80% of mentions",
                   (cutoff, mention_share.iloc[cutoff - 1]), xytext=(12, -32),
                   textcoords="offset points", color="#52514e")
    right.set_ylim(0, 1.02)
    right.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    right.set_xlabel("ingredients, most common first")
    right.set_title("Cumulative share of all ingredient mentions")

    fig.tight_layout()
    plt.show()

### Flavours as charts

**Left: share of teas with each flavour family.** Read it like the ingredient bars. A family that appears on a large share of teas, like `honey_sweet`, says little about any single tea.

**Right: how many flavour families a tea has.** One bar per count. Bar height is the share of all teas with exactly that many families. The bar at 0 is teas with no flavour data at all. For those teas, flavours can't help find similar teas, so only ingredients can.

In [ ]:
flavour_share = teas_per_item(flavours) / len(teas)
flavours_per_tea = flavours.groupby(level=0).size().reindex(teas.index, fill_value=0)
per_tea_share = flavours_per_tea.value_counts(normalize=True).sort_index()

with plt.rc_context(chart_style):
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 6))

    left.barh(flavour_share.index, flavour_share.to_numpy(), height=0.7, color=SERIES)
    left.invert_yaxis()
    left.grid(axis="y", visible=False)
    left.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    left.set_title("Share of teas with each flavour family")

    right.bar(per_tea_share.index, per_tea_share.to_numpy(), width=0.7, color=SERIES)
    right.grid(axis="x", visible=False)
    right.set_xticks(per_tea_share.index)
    right.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    right.set_xlabel("flavour families on one tea")
    right.set_title("How many flavour families a tea has")

    fig.tight_layout()
    plt.show()

## How many teas have identical recipes?

**Why it matters.** A model that only looks at ingredients sees two teas with the same ingredients as the same point. It gives them a perfect similarity score and can't rank one above the other. If many teas share a recipe, "most similar teas" turns into a long tie, and the order inside that tie is arbitrary.

**How to read the output.**
- The printed line is the share of teas whose exact set of ingredients also belongs to at least one other tea. Order doesn't matter: "Vanilla, Black tea leaf" and "Black tea leaf, Vanilla" are the same recipe.
- The table lists the most common recipes and how many teas have each one.

**What to look for.** Big groups with only one or two ingredients. Those teas need a second signal, such as flavours, to be told apart.

In [ ]:
recipe = ingredients.sort_values().groupby(level=0).agg(", ".join)
print(f"{recipe.duplicated(keep=False).mean():.0%} of teas share their exact recipe with another tea")
recipe.value_counts().head(10)

## Do related columns agree?

Three checks that the data is consistent with itself.

**1. Tea type by ingredient source.** A *crosstab* counts teas for every pair of values from two columns. Here each column adds up to 100%, so read down a column. For example: of the teas whose ingredients came from a composition list, what share is black, green, or missing? If the two columns look very different, the two kinds of shop are not describing the same mix of teas.

**2. Shops with the fewest usable teas.** The share of each shop's teas marked `usable_for_similarity = yes`, lowest first. A low share means most of that shop drops out of the model.

**3. Teas with the same name.** Some names appear more than once, usually at different shops. The printed line says how often teas that share a name also list the same ingredients. A low number means a shared name is not a shared recipe, so matching teas by name would be unreliable.

In [ ]:
display(
    pd.crosstab(teas["tea_type"].fillna("(missing)"), teas["source_quality"], normalize="columns")
    .style.format("{:.0%}")
)

display(
    teas["usable_for_similarity"].eq("yes").groupby(teas["shop"]).mean()
    .sort_values().head(8).to_frame("usable").style.format("{:.0%}")
)

names = teas["name"].str.strip().str.casefold()
same_name = teas[names.duplicated(keep=False)]
agree = same_name.groupby(names)["ingredients_english"].nunique(dropna=False).le(1).mean()
print(f"{len(same_name)} rows share a name with another row; the recipes agree in {agree:.0%} of those groups")

## Is an ingredient real, or a shop's boilerplate?

*Boilerplate* is text a shop repeats on every page, like a legal notice. If the text matching picks an ingredient out of it, every tea from that shop gets that ingredient by mistake.

**How to read the table.** One row per shop and ingredient:
- `n_teas`: how many of that shop's teas list the ingredient.
- `share_in_shop`: the same number as a fraction of that shop's teas. 100% means every tea.
- `share_elsewhere`: the fraction of teas at **all other shops** that list it.

Only the 25 rows with the highest `share_in_shop` are shown.

**What to look for.** A high `share_in_shop` next to a low `share_elsewhere` means one shop is very different from the rest. That is either a real specialty or a data error, and the numbers alone can't tell which. A Taiwanese oolong shop really does put oolong in almost every tea.

In [ ]:
shop_size = teas["shop"].value_counts()

pairs = ingredients.reset_index().drop_duplicates()
pairs["shop"] = teas.loc[pairs["index"], "shop"].to_numpy()

by_shop = pairs.value_counts(["shop", "ingredients_english"]).rename("n_teas").reset_index()
size = by_shop["shop"].map(shop_size)
by_shop["share_in_shop"] = by_shop["n_teas"] / size
by_shop["share_elsewhere"] = (
    (by_shop["ingredients_english"].map(ingredient_teas) - by_shop["n_teas"]) / (len(teas) - size)
)

by_shop.sort_values("share_in_shop", ascending=False).head(25).style.format(
    {"share_in_shop": "{:.0%}", "share_elsewhere": "{:.0%}"}
)

### Your rule

`looks_like_boilerplate` gets the three numbers from one row of the table above. It returns `True` if the row looks like boilerplate.

The cell then shows every row your rule flagged. Read that list and check two things: did it catch the errors, and did it leave the real specialties alone?

In [ ]:
def looks_like_boilerplate(n_teas: int, share_in_shop: float, share_elsewhere: float) -> bool:
    # TODO(human)
    raise NotImplementedError


suspicious = by_shop[[
    looks_like_boilerplate(row.n_teas, row.share_in_shop, row.share_elsewhere)
    for row in by_shop.itertuples()
]]
print(f"{len(suspicious)} of {len(by_shop)} shop-ingredient pairs look like boilerplate")
suspicious.sort_values("share_in_shop", ascending=False)